In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

table = pd.read_parquet("../data/processed/model_table.parquet")
FEATURES = [c for c in table if c not in ("cik", "end") and not c.startswith("y_")]
annee = table["end"].dt.year
table["split"] = np.select([annee <= 2019, annee <= 2021, annee <= 2024],
                           ["train", "val", "test"], default="prediction")

# événement (True/False)  ,  lignes où il est mesurable
EVENEMENTS = {
    "Capitaux propres : perte > 10 % des actifs":
        (table["y_E"] < -0.10,                       table["y_E"].notna()),
    "Trésorerie divisée par 2 ou plus":
        (table["y_C"] < -0.5 * table["C_A"],         table["y_C"].notna() & (table["C_A"] > 0)),
    "Actifs en baisse de plus de 20 %":
        (table["y_A"] < np.log(0.8),                 table["y_A"].notna()),
    "Capitaux propres deviennent négatifs":
        (table["E_A"] + table["y_E"] < 0,            table["y_E"].notna() & (table["E_A"] > 0)),
}

for nom, (evt, ok) in EVENEMENTS.items():
    freq = {s: round(evt[ok & (table["split"] == s)].mean(), 3) for s in ["train", "val", "test"]}
    print(f"{nom:<45} {freq}")

Capitaux propres : perte > 10 % des actifs    {'train': np.float64(0.159), 'val': np.float64(0.189), 'test': np.float64(0.2)}
Trésorerie divisée par 2 ou plus              {'train': np.float64(0.14), 'val': np.float64(0.204), 'test': np.float64(0.152)}
Actifs en baisse de plus de 20 %              {'train': np.float64(0.096), 'val': np.float64(0.123), 'test': np.float64(0.147)}
Capitaux propres deviennent négatifs          {'train': np.float64(0.027), 'val': np.float64(0.043), 'test': np.float64(0.038)}


In [2]:
def top_k(y, p, q=0.10):
    """Taux d'événements parmi les q % d'entreprises jugées les plus risquées."""
    top = p >= np.quantile(p, 1 - q)
    return y[top].mean()

resultats = []
for nom, (evt, ok) in EVENEMENTS.items():
    tr = table[ok & (table["split"] == "train")]
    va = table[ok & (table["split"] == "val")]
    y_tr, y_va = evt[tr.index].astype(int), evt[va.index].astype(int)

    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                         min_samples_leaf=100, l2_regularization=1.0,
                                         random_state=0)
    clf.fit(tr[FEATURES], y_tr)
    p = clf.predict_proba(va[FEATURES])[:, 1]

    taux_top = top_k(y_va.values, p)
    resultats.append({
        "événement": nom,
        "fréquence": y_va.mean(),
        "AUC modèle": roc_auc_score(y_va, p),
        "AUC règle « en perte »": roc_auc_score(y_va, va["perte"].fillna(0)),
        "AP modèle": average_precision_score(y_va, p),
        "taux dans le top 10 %": taux_top,
        "lift top 10 %": taux_top / y_va.mean(),
    })

pd.DataFrame(resultats).set_index("événement").round(3)

,fréquence,AUC modèle,AUC règle « en perte »,AP modèle,taux dans le top 10 %,lift top 10 %
événement,,,,,,
Capitaux propres : perte > 10 % des actifs,0.189,0.836,0.714,0.497,0.544,2.877
Trésorerie divisée par 2 ou plus,0.204,0.691,0.513,0.380,0.451,2.209
Actifs en baisse de plus de 20 %,0.123,0.808,0.672,0.324,0.363,2.957
Capitaux propres deviennent négatifs,0.043,0.890,0.598,0.237,0.215,5.026


In [3]:
from sklearn.inspection import permutation_importance

importances = {}
for nom, (evt, ok) in EVENEMENTS.items():
    tr = table[ok & (table["split"] == "train")]
    va = table[ok & (table["split"] == "val")]
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                         min_samples_leaf=100, l2_regularization=1.0,
                                         random_state=0)
    clf.fit(tr[FEATURES], evt[tr.index].astype(int))
    r = permutation_importance(clf, va[FEATURES], evt[va.index].astype(int),
                               scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1)
    importances[nom] = pd.Series(r.importances_mean, index=FEATURES)

imp = pd.DataFrame(importances)
for nom in imp:
    print(f"\n=== {nom} : top 8 ===")
    print(imp[nom].sort_values(ascending=False).head(8).round(4))


=== Capitaux propres : perte > 10 % des actifs : top 8 ===
ROA         0.0279
E_A         0.0149
taille      0.0116
CFO_A       0.0074
RE_A        0.0058
marge_op    0.0043
AC_LC       0.0041
d_L         0.0028
Name: Capitaux propres : perte > 10 % des actifs, dtype: float64

=== Trésorerie divisée par 2 ou plus : top 8 ===
d_C       0.0859
C_A       0.0207
PPE_A     0.0091
taille    0.0087
d_AC      0.0074
AC_A      0.0071
E_A       0.0062
CFF_A     0.0060
Name: Trésorerie divisée par 2 ou plus, dtype: float64

=== Actifs en baisse de plus de 20 % : top 8 ===
ROA         0.0158
taille      0.0150
CFO_A       0.0146
marge_op    0.0126
RE_A        0.0090
PPE_A       0.0072
E_A         0.0069
capex_A     0.0053
Name: Actifs en baisse de plus de 20 %, dtype: float64

=== Capitaux propres deviennent négatifs : top 8 ===
E_A           0.1570
RE_A          0.0246
ROA           0.0122
taille        0.0086
L_A           0.0064
C_A           0.0020
capex_A       0.0020
cout_dette    0.0016
Nam

In [4]:
top = {nom: imp[nom].sort_values(ascending=False).head(8) for nom in imp}
for nom, s in top.items():
    print(f"\n=== {nom} ===")
    print(s.round(4).to_string())


=== Capitaux propres : perte > 10 % des actifs ===
ROA         0.0279
E_A         0.0149
taille      0.0116
CFO_A       0.0074
RE_A        0.0058
marge_op    0.0043
AC_LC       0.0041
d_L         0.0028

=== Trésorerie divisée par 2 ou plus ===
d_C       0.0859
C_A       0.0207
PPE_A     0.0091
taille    0.0087
d_AC      0.0074
AC_A      0.0071
E_A       0.0062
CFF_A     0.0060

=== Actifs en baisse de plus de 20 % ===
ROA         0.0158
taille      0.0150
CFO_A       0.0146
marge_op    0.0126
RE_A        0.0090
PPE_A       0.0072
E_A         0.0069
capex_A     0.0053

=== Capitaux propres deviennent négatifs ===
E_A           0.1570
RE_A          0.0246
ROA           0.0122
taille        0.0086
L_A           0.0064
C_A           0.0020
capex_A       0.0020
cout_dette    0.0016


In [5]:
# Baselines « une seule variable » : est-ce que le modèle apporte plus qu'une règle simple ?
for nom, (evt, ok) in EVENEMENTS.items():
    va = table[ok & (table["split"] == "val")]
    y_va = evt[va.index].astype(int)
    print(f"\n=== {nom} ===")
    for f, sens in [("E_A", -1), ("ROA", -1), ("RE_A", -1), ("d_C", 1)]:
        score = sens * va[f].fillna(va[f].median())
        print(f"  règle « {f} {'faible' if sens < 0 else 'élevé'} » : AUC = {roc_auc_score(y_va, score):.3f}")


=== Capitaux propres : perte > 10 % des actifs ===
  règle « E_A faible » : AUC = 0.340
  règle « ROA faible » : AUC = 0.764
  règle « RE_A faible » : AUC = 0.772
  règle « d_C élevé » : AUC = 0.562

=== Trésorerie divisée par 2 ou plus ===
  règle « E_A faible » : AUC = 0.511
  règle « ROA faible » : AUC = 0.547
  règle « RE_A faible » : AUC = 0.546
  règle « d_C élevé » : AUC = 0.605

=== Actifs en baisse de plus de 20 % ===
  règle « E_A faible » : AUC = 0.409
  règle « ROA faible » : AUC = 0.722
  règle « RE_A faible » : AUC = 0.735
  règle « d_C élevé » : AUC = 0.547

=== Capitaux propres deviennent négatifs ===
  règle « E_A faible » : AUC = 0.779
  règle « ROA faible » : AUC = 0.693
  règle « RE_A faible » : AUC = 0.681
  règle « d_C élevé » : AUC = 0.490
